
# SpineXR — India vs Thailand Analysis Notebook 

This notebook performs a **clean, reproducible** analysis comparing India and Thailand clinician annotations for **Train** splits, with utilities to:
- Fix *"not correct"* **and** *"not sure"* items using explanations (when present), otherwise drop.
- Normalize image IDs consistently.
- Report **unique image** counts per split and **shared-image overlaps**.
- Compute **per-image** and **per-QA** correctness **agreements / discrepancies** (India vs Thailand).
- Provide **country-wise statistics**: how many questions/answers marked *wrong*.
- Count cases where an **answer was marked wrong** but the **(final) answer text says “normal / no finding”**.
- Extract **abnormality coverage** and count it **by unique images** (deduplicated) using keyword matching from questions/answers.
- Save helpful CSVs alongside plots/tables.

> **Paths** are configurable at the top. Run cells top-to-bottom.


In [ ]:

# =====================
# Config: file paths
# =====================
TRAIN_IN_PATH = "Final_CSV/Vindr_IN_train_final.csv"
TEST_IN_PATH  = "Final_CSV/Vindr_IN_test_final.csv"
TRAIN_TH_PATH = "Final_CSV/Vindr_Th_train_final.csv"
TEST_TH_PATH  = "Final_CSV/Vindr_Th_test_final.csv"


In [ ]:

# =====================
# Imports & Settings
# =====================
import os, re, json, math, itertools
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (6,4), "axes.grid": True})
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", 200)


In [ ]:

# ----------------------------------------------------
# Cleaning / correction helpers
# ----------------------------------------------------
def clean(s):
    s = str(s).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def _exists(p):
    try:
        return os.path.exists(p)
    except:
        return False

def normalize_image_id(s: object, drop_ext=False) -> str:
    s = str(s or "").strip()
    s = s.replace("\\", "/").split("/")[-1]   # basename
    s = s.split("?")[0]                        # drop query
    s = s.replace("\u200b", "")               # zero-width
    s = " ".join(s.split())                   # collapse spaces
    if drop_ext:
        s = re.sub(r"\.(png|jpg|jpeg|tif|tiff|bmp|webp)$", "", s, flags=re.I)
    return s.lower()

def norm_corr(s):
    s = str(s).strip().lower()
    s = s.replace("question not correct", "wrong").replace("answer not correct", "wrong")
    if s in {"correct","right","true","yes"}: return "correct"
    if s in {"wrong","incorrect","false","no"}: return "wrong"
    if s in {"not sure","unsure","uncertain"}: return "not sure"
    if s == "": return "blank"
    return s

def apply_corrections(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace question/answer with the explanation when marked 'not correct' **or 'not sure'**
    and an explanation is present. If marked 'not correct' or 'not sure' but **no** explanation,
    drop the row. Create audit flags for transparency.
    """
    df = df.copy()
    for col in ["question","answer","Question Correctness","Question Explanation",
                "Answer Correctness","Answer Explanation"]:
        if col not in df.columns:
            df[col] = ""

    # Always work on clean strings
    for c in ["question","answer","Question Correctness","Question Explanation",
              "Answer Correctness","Answer Explanation"]:
        df[c] = df[c].astype(str).apply(clean)

    df["Final Question"] = df["question"]
    df["Final Answer"]   = df["answer"]

    q_state = df["Question Correctness"].str.lower().str.strip()
    a_state = df["Answer Correctness"].str.lower().str.strip()

    q_fix = df["Question Explanation"].astype(str).str.strip()
    a_fix = df["Answer Explanation"].astype(str).str.strip()

    cond_q_bad_or_ns = q_state.str.contains("not correct|not sure", regex=True)
    cond_a_bad_or_ns = a_state.str.contains("not correct|not sure", regex=True)

    df["q_fixed_from_exp"] = False
    df["a_fixed_from_exp"] = False
    df["needs_manual_fix"] = False

    # Counts
    total_q_bad_ns = int(cond_q_bad_or_ns.sum())
    total_a_bad_ns = int(cond_a_bad_or_ns.sum())
    missing_q_exp  = int((cond_q_bad_or_ns & (q_fix == "")).sum())
    missing_a_exp  = int((cond_a_bad_or_ns & (a_fix == "")).sum())

    print("\n🩹 Correction summary (bad + not sure):")
    print(f"  - Q needing fix (bad or not sure): {total_q_bad_ns}")
    print(f"  - A needing fix (bad or not sure): {total_a_bad_ns}")
    print(f"  - Missing Q explanations: {missing_q_exp}")
    print(f"  - Missing A explanations: {missing_a_exp}")

    # Apply replacements when explanations exist
    mask_q = cond_q_bad_or_ns & (q_fix != "")
    mask_a = cond_a_bad_or_ns & (a_fix != "")
    df.loc[mask_q, "Final Question"] = q_fix[mask_q]
    df.loc[mask_q, "q_fixed_from_exp"] = True
    df.loc[mask_a, "Final Answer"] = a_fix[mask_a]
    df.loc[mask_a, "a_fixed_from_exp"] = True

    # Drop rows where 'bad or not sure' but no explanation exists
    before = len(df)
    drop_mask = (cond_q_bad_or_ns & (q_fix == "")) | (cond_a_bad_or_ns & (a_fix == ""))
    df["needs_manual_fix"] = drop_mask
    df = df[~drop_mask].copy()
    after = len(df)
    dropped = before - after
    print(f"  - Dropped rows with missing corrections: {dropped}")

    return df

def drop_not_sure(df: pd.DataFrame):
    """Drop rows where either Q or A correctness contains 'not sure'. Also return counts.
    Run this **after** apply_corrections so that fixed rows are kept."""
    q = df["Question Correctness"].astype(str).str.strip().str.lower()
    a = df["Answer Correctness"].astype(str).str.strip().str.lower()
    stats = {
        "Question not sure": int(q.str.contains("not sure").sum()),
        "Answer not sure":   int(a.str.contains("not sure").sum()),
        "Rows dropped (either not sure)": int((q.str.contains("not sure") | a.str.contains("not sure")).sum())
    }
    cleaned = df[~(q.str.contains("not sure") | a.str.contains("not sure"))].copy()
    return cleaned, stats

def max_cosine_per_test(X_test, X_train, block=2000):
    n = X_test.shape[0]
    out = np.zeros(n, dtype=np.float32)
    for i in range(0, n, block):
        sims = cosine_similarity(X_test[i:i+block], X_train)
        out[i:i+block] = sims.max(axis=1)
    return out


In [ ]:

# ----------------------------------------------------
# Load and preprocess all 4 CSVs
# ----------------------------------------------------
REQUIRED = ["image_id","category","question","answer",
            "Question Correctness","Question Explanation",
            "Answer Correctness","Answer Explanation"]

print("Files present:")
for p in [TRAIN_IN_PATH, TEST_IN_PATH, TRAIN_TH_PATH, TEST_TH_PATH]:
    print("  ", p, "✓" if _exists(p) else "✗")

def load_csv(path, country, split):
    df = pd.read_csv(path)
    missing = [c for c in REQUIRED if c not in df.columns]
    assert not missing, f"{path} missing columns: {missing}"
    df["country"] = country
    df["split"] = split
    for c in REQUIRED:
        df[c] = df[c].astype(str).apply(clean)
    return df[REQUIRED + ["country","split"]]

data = pd.concat([
    load_csv(TRAIN_IN_PATH, "India", "Train"),
    load_csv(TEST_IN_PATH,  "India", "Test"),
    load_csv(TRAIN_TH_PATH, "Thailand", "Train"),
    load_csv(TEST_TH_PATH,  "Thailand", "Test"),
], ignore_index=True)

pd.DataFrame({
    "Metric": ["Rows (raw)","Unique images (raw)","Unique categories (raw)"],
    "Value": [len(data), data["image_id"].nunique(), data["category"].nunique()]
})


In [ ]:

# --- Fix paths first ---
FILES = {
    ("India", "Train"):    TRAIN_IN_PATH,
    ("India", "Test"):     TEST_IN_PATH,
    ("Thailand", "Train"): TRAIN_TH_PATH,
    ("Thailand", "Test"):  TEST_TH_PATH,
}

# Normalize image ids consistently per split
rows = []
per_split_uniques = []
for (country, split), path in FILES.items():
    df = pd.read_csv(path)
    img_col = next((c for c in ["image_id","Image ID","image","Image"] if c in df.columns), None)
    assert img_col, f"No image id column in {path}: {list(df.columns)}"
    df["image_id_norm"] = df[img_col].apply(lambda x: normalize_image_id(x, drop_ext=False))
    df["country"] = country
    df["split"] = split
    rows.append(df[["image_id_norm","country","split"]])
    per_split_uniques.append({
        "Country": country,
        "Split": split,
        "Unique images (cleaned)": df["image_id_norm"].nunique(),
        "Rows": len(df),
        "Path": path
    })

per_split = pd.DataFrame(per_split_uniques).sort_values(["Country","Split"]).reset_index(drop=True)
all_df = pd.concat(rows, ignore_index=True)

print("Per-split (cleaned) unique images:")
display(per_split)

print("Global unique images across ALL splits (cleaned):", all_df["image_id_norm"].nunique())

# Overlap diagnostics across splits/countries
dup_ids = (all_df.groupby("image_id_norm")
                  .agg(n_splits=("split","nunique"),
                       n_rows=("split","size"))
                  .reset_index())
overlapping = dup_ids[dup_ids["n_splits"] > 1]
print("Images appearing in >1 split/country:", len(overlapping))

# Cross-tab sample of overlap
xt = pd.crosstab(all_df["image_id_norm"], all_df["country"] + " " + all_df["split"])
leak_candidates = xt[(xt > 0).sum(axis=1) > 1].head(20)
print("Sample overlap matrix (rows are image ids):")
display(leak_candidates)

# Within-country Train/Test leakage counts
leak_by_ctry = (all_df
    .assign(ctry_split=all_df["country"] + "|" + all_df["split"])
    .groupby(["image_id_norm","country"])["split"].nunique()
    .reset_index(name="splits_in_country"))
print("Images occurring in both Train and Test within the same country:",
      int((leak_by_ctry["splits_in_country"] > 1).sum()))


In [ ]:
# =============================================================
# Final Cleaning, Normalization & Statistics Summary
# =============================================================

import pandas as pd

# --- 1. Apply doctor-provided corrections ---
print("✅ Applying corrections to QA pairs...")
_data = apply_corrections(data)   # your existing function
_data["Question Correctness"] = _data["Question Correctness"].apply(norm_corr)
_data["Answer Correctness"]   = _data["Answer Correctness"].apply(norm_corr)

# --- 2. Drop residual "not sure" rows ---
print("\n✅ Dropping residual 'Not Sure' entries...")
_data, _not_sure_stats = drop_not_sure(_data)
print("Residual not-sure stats after correction:", _not_sure_stats)

# --- 3. Normalize image IDs (remove folder paths/extensions, etc.) ---
print("\n✅ Normalizing image IDs...")
_data["image_id_norm"] = _data["image_id"].apply(normalize_image_id)

# --- 4. Compute global statistics (after cleaning) ---
total_rows_cleaned = len(_data)
unique_images_cleaned = _data["image_id_norm"].nunique()

print(f"\n📊 Cleaned dataset summary:")
print(f"  - Total QA pairs (rows): {total_rows_cleaned}")
print(f"  - Unique images: {unique_images_cleaned}")

# --- 5. Split data by country/split ---
print("\n✅ Splitting data by country and split...")
ind_tr = _data[(_data["country"] == "India") & (_data["split"] == "Train")].copy()
ind_te = _data[(_data["country"] == "India") & (_data["split"] == "Test")].copy()
th_tr  = _data[(_data["country"] == "Thailand") & (_data["split"] == "Train")].copy()
th_te  = _data[(_data["country"] == "Thailand") & (_data["split"] == "Test")].copy()

# --- 6. Unique image counts per subset ---
subset_stats = pd.DataFrame([
    {"Country": "India", "Split": "Train", "Unique Images": ind_tr["image_id_norm"].nunique(), "QA Pairs": len(ind_tr)},
    {"Country": "India", "Split": "Test",  "Unique Images": ind_te["image_id_norm"].nunique(), "QA Pairs": len(ind_te)},
    {"Country": "Thailand", "Split": "Train", "Unique Images": th_tr["image_id_norm"].nunique(), "QA Pairs": len(th_tr)},
    {"Country": "Thailand", "Split": "Test",  "Unique Images": th_te["image_id_norm"].nunique(), "QA Pairs": len(th_te)},
])
print("\n📈 Per-country split summary:")
print(subset_stats.to_string(index=False))

# --- 7. Overlap between India-Train and Thailand-Train images ---
india_train_ids = set(ind_tr["image_id_norm"].unique())
thai_train_ids  = set(th_tr["image_id_norm"].unique())
common_train_ids = sorted(india_train_ids & thai_train_ids)
n_common_train = len(common_train_ids)

print(f"\n🔁 Common unique images between India-Train and Thailand-Train: {n_common_train}")

# --- 8. Verify no overlap between Train and Test sets ---
train_ids = set(_data[_data["split"] == "Train"]["image_id_norm"].unique())
test_ids  = set(_data[_data["split"] == "Test"]["image_id_norm"].unique())
overlap_train_test = sorted(train_ids & test_ids)

print(f"🚫 Common images between Train and Test sets: {len(overlap_train_test)}")

# --- 9. Final summary dictionary (for paper/logs) ---
final_summary = {
    "Total QA pairs (cleaned)": total_rows_cleaned,
    "Unique images (cleaned)": unique_images_cleaned,
    "Common India–Thailand Train images": n_common_train,
    "Train/Test overlap": len(overlap_train_test),
    "Residual 'not sure' rows dropped": _not_sure_stats.get("Rows dropped (either not sure)", 0)
}

print("\n✅ Final Dataset Summary:")
for k, v in final_summary.items():
    print(f"  {k:<40} {v}")

# --- Optional: Save summary tables for later use ---
subset_stats.to_csv("Final_CSV/dataset_split_summary.csv", index=False)
pd.DataFrame([final_summary]).to_csv("Final_CSV/dataset_global_summary.csv", index=False)


In [ ]:

# ---------------------------------------------------------------
# Align identical QAs (same image_id_norm + same question)
# ---------------------------------------------------------------
merged = ind_tr.merge(
    th_tr,
    on=["image_id_norm", "question"],
    suffixes=("_IN", "_TH"),
    how="inner"
)

print(f"Aligned QAs for comparison: {len(merged)} from {merged['image_id_norm'].nunique()} shared images")

# Discrepancy booleans
merged["question_correctness_diff"] = (merged["Question Correctness_IN"] != merged["Question Correctness_TH"])
merged["answer_correctness_diff"]   = (merged["Answer Correctness_IN"]   != merged["Answer Correctness_TH"])

# Per-image summary
diff_summary = (
    merged.groupby("image_id_norm")
          .agg(
              total_pairs=("question", "count"),
              q_corr_diff=("question_correctness_diff", "sum"),
              a_corr_diff=("answer_correctness_diff", "sum")
          )
          .reset_index()
)
diff_summary["percent_q_diff"] = (100 * diff_summary["q_corr_diff"] / diff_summary["total_pairs"]).round(2)
diff_summary["percent_a_diff"] = (100 * diff_summary["a_corr_diff"] / diff_summary["total_pairs"]).round(2)
diff_summary = diff_summary.sort_values("a_corr_diff", ascending=False).reset_index(drop=True)

print(f"Compared QA pairs for {len(diff_summary)} shared images (India vs Thailand Train)")
display(diff_summary.head(20))

# Save
Path("Final_CSV").mkdir(parents=True, exist_ok=True)
diff_summary.to_csv("Final_CSV/Train_IN_vs_TH_Correctness_Discrepancy_by_Image.csv", index=False)


In [ ]:

# =============================================================
# QA-level correctness agreement between India & Thailand
# =============================================================
qa_agreement = merged[[
    "image_id_norm", "question",
    "Final Answer_IN", "Final Answer_TH",
    "Question Correctness_IN", "Question Correctness_TH",
    "Answer Correctness_IN", "Answer Correctness_TH"
]].copy()

def answer_relation(i, t):
    if i == t:
        return f"Agree ({i})"
    if i == "correct" and t == "wrong":
        return "India✔ / Thai✘"
    if i == "wrong" and t == "correct":
        return "India✘ / Thai✔"
    return "mismatch"

def question_relation(i, t):
    if i == t:
        return f"Agree ({i})"
    if i == "correct" and t == "wrong":
        return "India✔ / Thai✘"
    if i == "wrong" and t == "correct":
        return "India✘ / Thai✔"
    return "mismatch"

qa_agreement["Answer Agreement"]   = [answer_relation(i, t)   for i, t in zip(qa_agreement["Answer Correctness_IN"], qa_agreement["Answer Correctness_TH"])]
qa_agreement["Question Agreement"] = [question_relation(i, t) for i, t in zip(qa_agreement["Question Correctness_IN"], qa_agreement["Question Correctness_TH"])]

# Summaries
ans_summary = qa_agreement["Answer Agreement"].value_counts().reset_index()
ans_summary.columns = ["Relation", "Count"]
ques_summary = qa_agreement["Question Agreement"].value_counts().reset_index()
ques_summary.columns = ["Relation", "Count"]

print("Answer correctness relation summary:")
display(ans_summary)
print("Question correctness relation summary:")
display(ques_summary)

# Plots (matplotlib only)
def plot_bar(df, title):
    plt.figure()
    plt.bar(df["Relation"], df["Count"])
    plt.title(title)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

plot_bar(ans_summary, "Answer Correctness — India vs Thailand Agreement")
plot_bar(ques_summary, "Question Correctness — India vs Thailand Agreement")


In [ ]:

# =============================================================
# Country-wise wrong counts (Q and A) — All splits and Train-only
# =============================================================
def wrong_counts(df):
    out = (df.assign(Q=lambda d: d["Question Correctness"].apply(lambda x: str(x).lower() == "wrong"),
                     A=lambda d: d["Answer Correctness"].apply(lambda x: str(x).lower() == "wrong"))
             .groupby(["country","split"])
             .agg(Q_wrong=("Q","sum"), A_wrong=("A","sum"), Rows=("Q","size"))
             .reset_index())
    out["Q_wrong_rate_%"] = (100 * out["Q_wrong"] / out["Rows"]).round(2)
    out["A_wrong_rate_%"] = (100 * out["A_wrong"] / out["Rows"]).round(2)
    return out

wrong_all = wrong_counts(_data)
print("Wrong counts by country & split (after corrections):")
display(wrong_all)

wrong_train = wrong_counts(_data[_data["split"]=="Train"])
print("Wrong counts for Train only:")
display(wrong_train)


In [ ]:

# =============================================================
# Wrong Answer but Final Answer says "normal/no finding" (by country & split)
# =============================================================
NORMAL_TERMS = [
    r"\bnormal\b",
    r"no finding",
    r"no abnormalit(y|ies)",
    r"unremarkable",
    r"within normal limits",
    r"wnl"
]
normal_re = re.compile("|".join(NORMAL_TERMS), flags=re.I)

_data["Final Answer"] = _data["Final Answer"].astype(str)

cond_wrong_A = _data["Answer Correctness"].str.lower().eq("wrong")
cond_normal_text = _data["Final Answer"].apply(lambda s: bool(normal_re.search(str(s))))

wrong_normal = (_data[cond_wrong_A]
                .assign(is_normal=cond_normal_text[cond_wrong_A].values)
                .groupby(["country","split"])
                .agg(WrongA=("Answer Correctness", "size"),
                     WrongA_with_Normal=("is_normal","sum"))
                .reset_index())
wrong_normal["Rate_%"] = (100 * wrong_normal["WrongA_with_Normal"] / wrong_normal["WrongA"].replace(0, np.nan)).round(2)

print("Wrong-Answer cases where (final) answer text indicates NORMAL / NO FINDING:")
display(wrong_normal)


In [ ]:

# =============================================================
# Abnormality coverage (unique images) from Questions + Final Answers
# =============================================================
ABN_PATTERNS = {
    "Spondylolisthesis": r"\bspondylolisthes(is|es)\b|\blisthesis\b",
    "Disc space narrowing": r"disc (space )?narrowing|\bnarrowing of disc\b",
    "Osteophytes": r"\bosteophyte(s)?\b|bony spur(s)?",
    "Vertebral collapse": r"\b(collapse|compression) fracture\b|\bvertebral collapse\b|\bcompression\b",
    "Foraminal stenosis": r"\bforaminal stenosis\b|\bforamen stenosis\b",
    "Scoliosis": r"\bscoliosis\b|\bscoliotic\b",
    "Kyphosis": r"\bkyphosis\b|\bkyphotic\b",
    "Lordosis": r"\blordosis\b|\blordotic\b",
    "Fracture": r"\bfracture\b|\bfx\b",
    "Implant/Hardware": r"implant|fixation|pedicle screw|rod|cage|plate|hardware|fusion device",
    "Degeneration": r"degenerative|spondylosis|degeneration",
    "Alignment issue": r"malalignment|misalignment|alignment abnormality|anterolisthesis|retrolisthesis",
    "No finding / Normal": r"no finding|no abnormal|\bnormal\b|unremarkable",
}

def build_abn_table(df):
    df = df.copy()
    df["text_all"] = (df["question"].astype(str) + " " + df["Final Answer"].astype(str)).str.lower()
    rows = []
    for (ct, sp), g in df.groupby(["country","split"]):
        img_text = g.groupby("image_id_norm")["text_all"].apply(lambda s: " ".join(s)).reset_index()
        total_imgs = img_text["image_id_norm"].nunique()
        for label, pat in ABN_PATTERNS.items():
            pat_re = re.compile(pat, flags=re.I)
            count_imgs = int(img_text["text_all"].apply(lambda t: bool(pat_re.search(t))).sum())
            rows.append({"Country": ct, "Split": sp, "Abnormality": label,
                         "UniqueImagesWithAbn": count_imgs, "TotalUniqueImages": total_imgs,
                         "Coverage_%": round(100 * count_imgs / total_imgs, 2) if total_imgs else np.nan})
    return pd.DataFrame(rows).sort_values(["Country","Split","Abnormality"]).reset_index(drop=True)

abn_by_image = build_abn_table(_data)
print("Abnormality coverage by UNIQUE images (deduped per image):")
display(abn_by_image.head(20))

from pathlib import Path
Path("Final_CSV").mkdir(parents=True, exist_ok=True)
abn_by_image.to_csv("Final_CSV/Abnormality_Coverage_By_Unique_Images.csv", index=False)


In [ ]:

# =============================================================
# 
# Shared-image statistics (India–Thailand Train)
# =============================================================
shared_ind_th = _data[_data["image_id_norm"].isin(common_ids) & _data["split"].eq("Train")]
shared_stats = (shared_ind_th.groupby(["country"])
                .agg(UniqueImages=("image_id_norm","nunique"),
                     QAs=("image_id_norm","size"))
                .reset_index())

summary_tbl = pd.DataFrame({
    "Total unique images (India Train)": [len(india_ids)],
    "Total unique images (Thailand Train)": [len(thai_ids)],
    "Shared unique images (Train)": [len(common_ids)],
    "Aligned QA pairs on shared images": [len(merged)]
})
print("Shared image / QA alignment summary:")
display(summary_tbl)
display(shared_stats)


In [ ]:

qa_out = qa_agreement.copy()
qa_out.to_csv("Final_CSV/QA_Agreement_India_vs_Thailand.csv", index=False)
print("Saved: Final_CSV/QA_Agreement_India_vs_Thailand.csv")



## Notes & Next Steps

- You can extend `ABN_PATTERNS` to include finer-grained terms used by your clinicians.
- For **per-category** breakdowns, group by `category`.
- For **per-abnormality confusion trends**, map each QA to an abnormality label using regex hits on question/answer text and then aggregate agreement relations.
- CSVs written under `Final_CSV/`:
  - `Train_IN_vs_TH_Correctness_Discrepancy_by_Image.csv`
  - `QA_Agreement_India_vs_Thailand.csv`
  - `Abnormality_Coverage_By_Unique_Images.csv`


In [ ]:
import pandas as pd
import re
from pathlib import Path

# Folder and filenames to clean
CSV_PATHS = [
    "Final_CSV/Vindr_IN_train_final.csv",
    "Final_CSV/Vindr_IN_test_final.csv",
    "Final_CSV/Vindr_Th_train_final.csv",
    "Final_CSV/Vindr_Th_test_final.csv",
]

# Regex patterns for detecting "Answer" (with or without colon/space)
ANSWER_PAT = re.compile(r"\banswer\s*:?\s*", flags=re.IGNORECASE)

def clean_answer_text(text: str) -> str:
    """Cleans 'Answer:' or 'Answer' tokens.
       - If appears at start -> remove it only.
       - If appears mid-text -> truncate at that point.
    """
    if not isinstance(text, str):
        return text
    s = text.strip()

    # Case 1: starts with 'Answer:' or similar
    if re.match(r"^\s*answer\s*:?\s*", s, flags=re.IGNORECASE):
        s = re.sub(r"^\s*answer\s*:?\s*", "", s, flags=re.IGNORECASE).strip()
        return s

    # Case 2: appears later in text — truncate
    match = re.search(r"\banswer\s*:?\s*", s, flags=re.IGNORECASE)
    if match:
        s = s[:match.start()].strip()
    return s

# Loop through CSVs
for path in CSV_PATHS:
    p = Path(path)
    if not p.exists():
        print(f"⚠️ File not found: {p}")
        continue

    print(f"Cleaning file: {p.name}")
    df = pd.read_csv(p)

    if "answer" not in df.columns:
        print(f"  ❌ No 'answer' column in {p.name}")
        continue

    before_samples = df["answer"].head(3).tolist()

    # Clean answers
    df["answer"] = df["answer"].astype(str).apply(clean_answer_text)

    after_samples = df["answer"].head(3).tolist()

    print("  ✅ Cleaned successfully. Sample before/after:")
    for b, a in zip(before_samples, after_samples):
        print("   ", f"Before: {b[:60]}")
        print("   ", f"After : {a[:60]}")
        print("   ", "-"*40)

    # Save new CSV (you can overwrite or append suffix)
    new_path = p.with_name(p.stem + "_fixed.csv")
    df.to_csv(new_path, index=False)
    print(f"  💾 Saved cleaned file to: {new_path}\n")


In [ ]:
import pandas as pd
from pathlib import Path

# Paths to the cleaned CSVs
CSV_PATHS = [
    "Final_CSV/Vindr_IN_train_final_fixed.csv",
    "Final_CSV/Vindr_IN_test_final_fixed.csv",
    "Final_CSV/Vindr_Th_train_final_fixed.csv",
    "Final_CSV/Vindr_Th_test_final_fixed.csv",
]

results = []

def text_lengths(s: str):
    """Return (char_len, word_len) safely."""
    if not isinstance(s, str):
        s = str(s)
    s = s.strip()
    return len(s), len(s.split())

for path in CSV_PATHS:
    p = Path(path)
    if not p.exists():
        print(f"⚠️ File not found: {p}")
        continue

    print(f"Analyzing file: {p.name}")
    df = pd.read_csv(p)

    # Check required columns
    if not {"question", "answer"} <= set(df.columns):
        print(f"  ❌ Missing 'question' or 'answer' columns.")
        continue

    # Compute lengths
    df["q_char_len"], df["q_word_len"] = zip(*df["question"].astype(str).map(text_lengths))
    df["a_char_len"], df["a_word_len"] = zip(*df["answer"].astype(str).map(text_lengths))

    summary = {
        "File": p.name,
        "Q_min_chars": int(df["q_char_len"].min()),
        "Q_max_chars": int(df["q_char_len"].max()),
        "Q_min_words": int(df["q_word_len"].min()),
        "Q_max_words": int(df["q_word_len"].max()),
        "A_min_chars": int(df["a_char_len"].min()),
        "A_max_chars": int(df["a_char_len"].max()),

        
        
        "A_min_words": int(df["a_word_len"].min()),
        "A_max_words": int(df["a_word_len"].max()),
        "Total_Rows": len(df)
    }
    results.append(summary)

# Convert to summary DataFrame
summary_df = pd.DataFrame(results)
print("\n🔹 Question & Answer Length Summary:")
display(summary_df)

# Optionally save to CSV
out_path = Path("Final_CSV/QnA_Length_Stats.csv")
summary_df.to_csv(out_path, index=False)
print(f"\n💾 Saved stats to: {out_path}")



In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Try importing textstat; if not available, use NaN fallbacks.
try:
    import textstat
    _HAS_TEXTSTAT = True
except Exception:
    _HAS_TEXTSTAT = False

# -------------------------------------------------
# Config: cleaned CSVs produced in the previous step
# -------------------------------------------------
CSV_PATHS = [
    "Final_CSV/Vindr_IN_train_final_fixed.csv",
    "Final_CSV/Vindr_IN_test_final_fixed.csv",
    "Final_CSV/Vindr_Th_train_final_fixed.csv",
    "Final_CSV/Vindr_Th_test_final_fixed.csv",
]

# -----------------------------------------
# Lightweight tokenizer & helper functions
# -----------------------------------------
_WORD_RE = re.compile(r"[A-Za-z]+")

# Minimal English stopword list (no external downloads)
_STOPWORDS = {
    "a","an","the","and","or","but","if","while","of","to","in","on","for","with","at","by","from",
    "is","am","are","was","were","be","been","being","do","does","did","doing","have","has","had",
    "having","this","that","these","those","it","its","as","than","then","so","such","into","over",
    "under","between","through","during","before","after","above","below","up","down","out","off",
    "again","further","once","here","there","when","where","why","how","all","any","both","each",
    "few","more","most","other","some","no","nor","not","only","own","same","too","very","can",
    "will","just","don","should","now"
}

def tokenize(text):
    """Regex-based word tokenizer (letters only), lowercased."""
    return [w.lower() for w in _WORD_RE.findall(str(text))]

def count_words(text):
    return len(tokenize(text))

def ttr(tokens):
    return (len(set(tokens)) / len(tokens)) if tokens else np.nan

def mtld(tokens, threshold=0.72):
    """MTLD implementation without external deps."""
    if not tokens:
        return np.nan
    factors = 0
    types = set()
    run_len = 0
    for w in tokens:
        run_len += 1
        types.add(w)
        if len(types) / run_len < threshold:
            factors += 1
            types = set()
            run_len = 0
    # partial factor
    if run_len > 0:
        # proportion of remaining segment relative to threshold
        remainder = (len(types) / run_len - threshold) / (1 - threshold)
        # if remainder negative, count as full factor; else partial
        factors += 1 if remainder <= 0 else (1 - remainder)
    return (len(tokens) / factors) if factors > 0 else np.nan

def lexical_density(tokens):
    """Approx content-word ratio using non-stopwords as proxy."""
    if not tokens:
        return np.nan
    content = [w for w in tokens if w not in _STOPWORDS]
    return len(content) / len(tokens) if tokens else np.nan

def readability_scores(text):
    """Readability via textstat if available."""
    if not _HAS_TEXTSTAT:
        return dict(FRE=np.nan, FKG=np.nan, GFI=np.nan)
    text = str(text)
    return dict(
        FRE = textstat.flesch_reading_ease(text),
        FKG = textstat.flesch_kincaid_grade(text),
        GFI = textstat.gunning_fog(text),
    )

# -----------------------------------------
# Per-file computation
# -----------------------------------------
rows = []

for path in CSV_PATHS:
    p = Path(path)
    if not p.exists():
        print(f"⚠️ Missing file: {p}")
        continue

    df = pd.read_csv(p)
    if not {"question","answer"} <= set(df.columns):
        print(f"❌ {p.name} missing 'question'/'answer' columns")
        continue

    # Infer split & country from filename (robust-ish)
    fname = p.name.lower()
    split = "Train" if "train" in fname else ("Test" if "test" in fname else "Unknown")
    country = "India" if "_in_" in fname else ("Thailand" if "_th_" in fname else "Unknown")

    # Tokenize all questions/answers
    q_tokens_list = df["question"].astype(str).map(tokenize).tolist()
    a_tokens_list = df["answer"].astype(str).map(tokenize).tolist()

    # Flatten per file
    q_all = [w for toks in q_tokens_list for w in toks]
    a_all = [w for toks in a_tokens_list for w in toks]
    all_all = q_all + a_all

    # Lengths (mean words)
    mean_q_words = float(np.mean([len(t) for t in q_tokens_list])) if len(q_tokens_list) else np.nan
    mean_a_words = float(np.mean([len(t) for t in a_tokens_list])) if len(a_tokens_list) else np.nan

    # Diversity / density
    q_ttr = ttr(q_all); a_ttr = ttr(a_all); all_ttr = ttr(all_all)
    q_mtld = mtld(q_all); a_mtld = mtld(a_all); all_mtld = mtld(all_all)
    q_ld = lexical_density(q_all); a_ld = lexical_density(a_all); all_ld = lexical_density(all_all)

    # Readability on concatenated text (more stable)
    joined_text = " ".join(df["question"].astype(str).tolist() + df["answer"].astype(str).tolist())
    rb = readability_scores(joined_text)

    rows.append({
        "File": p.name,
        "Country": country,
        "Split": split,
        "Total_Rows": len(df),

        # Basic length
        "Mean_Q_words": round(mean_q_words, 2),
        "Mean_A_words": round(mean_a_words, 2),

        # TTR
        "Q_TTR": round(q_ttr, 4) if q_ttr==q_ttr else np.nan,
        "A_TTR": round(a_ttr, 4) if a_ttr==a_ttr else np.nan,
        "All_TTR": round(all_ttr, 4) if all_ttr==all_ttr else np.nan,

        # MTLD
        "Q_MTLD": round(q_mtld, 2) if q_mtld==q_mtld else np.nan,
        "A_MTLD": round(a_mtld, 2) if a_mtld==a_mtld else np.nan,
        "All_MTLD": round(all_mtld, 2) if all_mtld==all_mtld else np.nan,

        # Lexical Density
        "Q_LexDensity": round(q_ld, 3) if q_ld==q_ld else np.nan,
        "A_LexDensity": round(a_ld, 3) if a_ld==a_ld else np.nan,
        "All_LexDensity": round(all_ld, 3) if all_ld==all_ld else np.nan,

        # Readability
        "Flesch_Reading_Ease": rb["FRE"],
        "Flesch_Kincaid_Grade": rb["FKG"],
        "Gunning_Fog_Index": rb["GFI"],
    })

lex_df = pd.DataFrame(rows)
print("🔹 Per-file lexical complexity:")
display(lex_df)

# -----------------------------------------
# Aggregations: Train vs Test and Overall
# -----------------------------------------
def safe_mean(df, cols):
    return {k: df[k].mean(numeric_only=True) for k in cols if k in df.columns}

# Columns to average
metric_cols = [
    "Mean_Q_words","Mean_A_words",
    "Q_TTR","A_TTR","All_TTR",
    "Q_MTLD","A_MTLD","All_MTLD",
    "Q_LexDensity","A_LexDensity","All_LexDensity",
    "Flesch_Reading_Ease","Flesch_Kincaid_Grade","Gunning_Fog_Index"
]

# Train vs Test
split_summary = (lex_df.groupby("Split", dropna=False)[metric_cols]
                 .mean(numeric_only=True)
                 .reset_index())
print("\n🔹 Train vs Test summary (mean across files):")
display(split_summary)

# Overall
overall_summary = pd.DataFrame([lex_df[metric_cols].mean(numeric_only=True)])
overall_summary.insert(0, "Scope", "Overall (All Files)")
print("\n🔹 Overall summary (mean across all files):")
display(overall_summary)

# Optional: Country-level (India vs Thailand) if desired
country_summary = (lex_df.groupby("Country", dropna=False)[metric_cols]
                   .mean(numeric_only=True)
                   .reset_index())
print("\n🔹 Country summary (mean across files):")
display(country_summary)

# Save all summaries
out_dir = Path("Final_CSV")
out_dir.mkdir(parents=True, exist_ok=True)
lex_df.to_csv(out_dir / "Lexical_Complexity_PerFile.csv", index=False)
split_summary.to_csv(out_dir / "Lexical_Complexity_TrainVsTest.csv", index=False)
overall_summary.to_csv(out_dir / "Lexical_Complexity_Overall.csv", index=False)
country_summary.to_csv(out_dir / "Lexical_Complexity_ByCountry.csv", index=False)
print("\n💾 Saved: Lexical_Complexity_PerFile.csv, _TrainVsTest.csv, _Overall.csv, _ByCountry.csv")


In [ ]:
!pip install textstat -qqq


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# -------------------------------
# Load metrics
# -------------------------------
lex_file = Path("Final_CSV/Lexical_Complexity_PerFile.csv")
split_file = Path("Final_CSV/Lexical_Complexity_TrainVsTest.csv")
country_file = Path("Final_CSV/Lexical_Complexity_ByCountry.csv")

lex_df = pd.read_csv(lex_file)
split_df = pd.read_csv(split_file)
country_df = pd.read_csv(country_file)

# -------------------------------
# Normalize ordering
# -------------------------------
if "Split" in split_df.columns:
    split_df["Split"] = pd.Categorical(split_df["Split"], ["Train", "Test"], ordered=True)
    split_df = split_df.sort_values("Split")

if "Country" in country_df.columns:
    country_df["Country"] = pd.Categorical(country_df["Country"], ["India", "Thailand"], ordered=True)
    country_df = country_df.sort_values("Country")

# -------------------------------
# Output directory
# -------------------------------
fig_dir = Path("Final_CSV/Figures")
fig_dir.mkdir(parents=True, exist_ok=True)

# =====================================================
# 1️⃣ Mean Question / Answer Length — STACKED
# =====================================================

def infer_country_split(fname: str):
    s = fname.lower()
    country = "Thailand" if ("th_" in s or "_th_" in s) else "India"
    split = "Train" if "train" in s else "Test"
    return country, split

# Build tmp dataframe
tmp = lex_df.copy()
tmp[["Country", "Split"]] = tmp["File"].apply(
    lambda f: pd.Series(infer_country_split(f))
)

# Enforce order
tmp["Country"] = pd.Categorical(tmp["Country"], ["India", "Thailand"], ordered=True)
tmp["Split"]   = pd.Categorical(tmp["Split"], ["Train", "Test"], ordered=True)
tmp = tmp.sort_values(["Country", "Split"]).reset_index(drop=True)

# X positions and labels
labels = (tmp["Country"].astype(str) + "_" + tmp["Split"].astype(str)).tolist()
x = np.arange(len(tmp))

# Font sizes
AXIS_LABEL_FONTSIZE = 12
TICK_FONTSIZE = 12
TITLE_FONTSIZE = 13
LEGEND_FONTSIZE = 12

plt.figure(figsize=(8, 4))

plt.bar(x, tmp["Mean_Q_words"], color="#4C72B0", label="Question")
plt.bar(
    x,
    tmp["Mean_A_words"],
    bottom=tmp["Mean_Q_words"],
    color="#FFA500",
    label="Answer"
)

# Annotate totals
totals = tmp["Mean_Q_words"].values + tmp["Mean_A_words"].values
for xi, total in zip(x, totals):
    plt.text(xi, total + 0.5, f"{total:.0f}",
             ha="center", va="bottom", fontsize=9)

plt.xticks(x, labels, rotation=20, ha="right", fontsize=TICK_FONTSIZE)
plt.yticks(fontsize=TICK_FONTSIZE)
plt.ylabel("Mean Words", fontsize=AXIS_LABEL_FONTSIZE)
plt.xlabel("Country–Split", fontsize=AXIS_LABEL_FONTSIZE)
plt.title("Mean Words (Stacked): Question + Answer", fontsize=TITLE_FONTSIZE)
plt.legend(fontsize=LEGEND_FONTSIZE)

plt.tight_layout()
plt.savefig(fig_dir / "mean_word_length_per_file_stacked.png", dpi=300)
plt.show()

# ======================================================
# 2️⃣ Vocabulary Richness — TTR vs MTLD
# ======================================================

plt.figure(figsize=(7, 4))

plt.scatter(lex_df["Q_TTR"], lex_df["Q_MTLD"],
            color="tab:blue", label="Question",
            s=60, edgecolor="k", zorder=3)

plt.scatter(lex_df["A_TTR"], lex_df["A_MTLD"],
            color="tab:orange", label="Answer",
            s=60, edgecolor="k", zorder=3)

Q_OFFSETS = {
    "IN-Train": (6, 6), "TH-Train": (10, -10),
    "IN-Test": (5, 5), "TH-Test": (3, 3),
}

A_OFFSETS = {
    "IN-Train": (-12, -15), "TH-Train": (-10, 8),
    "IN-Test": (-12, -15), "TH-Test": (-25, -15),
}

for i, name in enumerate(lex_df["File"]):

    label = (
        name.replace("Vindr_", "")
            .replace("_final_fixed.csv", "")
            .replace("IN_", "IN-")
            .replace("Th_", "TH-")
            .replace("_train", "-Train")
            .replace("_test", "-Test")
    )
    label = label.replace("train", "Train").replace("test", "Test")

    q_dx, q_dy = Q_OFFSETS.get(label, (6, 6))
    a_dx, a_dy = A_OFFSETS.get(label, (-8, -8))

    plt.annotate(label,
        xy=(lex_df["Q_TTR"].iloc[i], lex_df["Q_MTLD"].iloc[i]),
        xytext=(q_dx, q_dy),
        textcoords="offset points",
        fontsize=12, color="tab:blue", alpha=0.85,
        arrowprops=dict(arrowstyle="-", lw=0.5, alpha=0.6))

    plt.annotate(label,
        xy=(lex_df["A_TTR"].iloc[i], lex_df["A_MTLD"].iloc[i]),
        xytext=(a_dx, a_dy),
        textcoords="offset points",
        fontsize=12, color="tab:orange", alpha=0.85,
        arrowprops=dict(arrowstyle="-", lw=0.5, alpha=0.6))

plt.xlabel("Type–Token Ratio (TTR)", fontsize=12)
plt.ylabel("MTLD (Lexical Diversity)", fontsize=12)
plt.title("Vocabulary Richness — TTR vs MTLD (IN/TH × Train/Test)", fontsize=13)
plt.legend(fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(fig_dir / "lexical_richness_scatter.png", dpi=300)
plt.show()

# ======================================================
# 3️⃣ Lexical Density Comparison (Train vs Test)
# ======================================================

plt.figure(figsize=(6, 4))

plt.bar(
    split_df["Split"],
    split_df["All_LexDensity"],
    color=["#F2A06E", "#82CA81"]
)

plt.xlabel("Split", fontsize=12)
plt.ylabel("Average Lexical Density", fontsize=12)
plt.title("Lexical Density — Train vs Test", fontsize=13)

plt.ylim(0, 1)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

plt.tight_layout()
plt.savefig(fig_dir / "lexical_density_train_test.png", dpi=300)
plt.show()


# ======================================================
# 4️⃣ Lexical Density Comparison (India vs Thailand)
# ======================================================

plt.figure(figsize=(6, 4))

plt.bar(
    country_df["Country"],
    country_df["All_LexDensity"],
    color=["#8172B2", "#CCB974"]
)

plt.xlabel("Country", fontsize=12)
plt.ylabel("Average Lexical Density", fontsize=12)
plt.title("Lexical Density — India vs Thailand", fontsize=13)

plt.ylim(0, 1)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

plt.tight_layout()
plt.savefig(fig_dir / "lexical_density_country.png", dpi=300)
plt.show()


# ======================================================
# 5️⃣ Readability Indices Comparison (Train vs Test)
# ======================================================

cols = [
    "Flesch_Reading_Ease",
    "Flesch_Kincaid_Grade",
    "Gunning_Fog_Index"
]

labels = ["Flesch Ease", "FK Grade", "Gunning Fog"]
x = np.arange(len(cols))
width = 0.35

train_row = split_df[split_df["Split"] == "Train"][cols]
test_row  = split_df[split_df["Split"] == "Test"][cols]

train_vals = train_row.iloc[0].values if len(train_row) else [0, 0, 0]
test_vals  = test_row.iloc[0].values  if len(test_row)  else [0, 0, 0]

plt.figure(figsize=(7, 4))

plt.bar(
    x - width / 2,
    train_vals,
    width,
    label="Train",
    color="#F2A06E"
)

plt.bar(
    x + width / 2,
    test_vals,
    width,
    label="Test",
    color="#55A868"
)

plt.xticks(x, labels, fontsize=11)
plt.ylabel("Score", fontsize=12)
plt.title("Readability Comparison — Train vs Test", fontsize=13)
plt.legend(fontsize=11)

plt.tight_layout()
plt.savefig(fig_dir / "readability_train_test.png", dpi=300)
plt.show()


# ======================================================
# 6️⃣ Readability Indices Comparison (India vs Thailand)
# ======================================================

ind_row = country_df[country_df["Country"] == "India"][cols]
th_row  = country_df[country_df["Country"] == "Thailand"][cols]

ind_vals = ind_row.iloc[0].values if len(ind_row) else [0, 0, 0]
th_vals  = th_row.iloc[0].values  if len(th_row)  else [0, 0, 0]

plt.figure(figsize=(7, 4))

plt.bar(
    x - width / 2,
    ind_vals,
    width,
    label="India",
    color="#8172B2"
)

plt.bar(
    x + width / 2,
    th_vals,
    width,
    label="Thailand",
    color="#82CA81"
)

plt.xticks(x, labels, fontsize=11)
plt.ylabel("Score", fontsize=12)
plt.title("Readability Comparison — India vs Thailand", fontsize=13)
plt.legend(fontsize=11)

plt.tight_layout()
plt.savefig(fig_dir / "readability_country.png", dpi=300)
plt.show()

print(f"✅ All lexical complexity figures saved in: {fig_dir.resolve()}")


print(f"✅ All lexical complexity figures saved in: {fig_dir.resolve()}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

# ==== data (India–Test) ====
data = {
    "Abnormality": [
        "Alignment issue","Degeneration","Disc space narrowing","Foraminal stenosis",
        "Fracture","Implant/Hardware","Kyphosis","Lordosis","No finding / Normal",
        "Osteophytes","Scoliosis","Spondylolisthesis","Vertebral collapse"
    ],
    "Coverage_%": [0.00,32.67,7.33,2.67,28.67,4.00,1.33,0.67,44.67,31.33,4.00,3.33,14.67]
}
df = pd.DataFrame(data)

# ==== group small categories (≤10%) ====
thr = 10.0
major = df[df["Coverage_%"] > thr].copy()
minor = df[df["Coverage_%"] <= thr].copy()

other_pct = minor["Coverage_%"].sum()
other_names = minor["Abnormality"].tolist()

# final plot table
plot_df = pd.concat(
    [major, pd.DataFrame([{"Abnormality": "Other (<10%)", "Coverage_%": other_pct}])],
    ignore_index=True
).sort_values("Coverage_%", ascending=False)

# normalize
plot_df["Coverage_%"] = 100 * plot_df["Coverage_%"] / plot_df["Coverage_%"].sum()

# ==== figure ====
fig, ax = plt.subplots(figsize=(9.5, 6.0))
ax.set_aspect("equal")

colors = cm.Pastel1.colors[:len(plot_df)]
wedges, _, autopcts = ax.pie(
    plot_df["Coverage_%"],
    startangle=130,
    colors=colors,
    autopct="%1.1f%%",
    pctdistance=0.75,
    wedgeprops=dict(edgecolor="white", linewidth=1.2)
)

for t in autopcts:
    t.set_fontsize(12)

# slice labels (only main categories)
labels = plot_df["Abnormality"].tolist()
bbox_props = dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", lw=0.8)
arrow_props = dict(arrowstyle="-", lw=0.8, color="0.4")

for w, lab in zip(wedges, labels):
    ang = (w.theta2 + w.theta1) / 2
    x, y = np.cos(np.deg2rad(ang)), np.sin(np.deg2rad(ang))
    ax.annotate(
        lab,
        xy=(x * 0.95, y * 0.95),
        xytext=(1.12 * np.sign(x), 1.12 * y),
        ha="left" if x > 0 else "right",
        va="center",
        bbox=bbox_props,
        arrowprops=arrow_props
    )

# ==== RIGHT-SIDE LEGEND FOR "OTHER" ====
other_text = "\n".join(f"• {name}" for name in other_names)

fig.text(
    0.85, 0.25,
    "Other (<10%)\n" + other_text,
    ha="left", va="center",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.8")
)

ax.set_title("Abnormality Coverage in SpineXR-VQA", fontsize=13, pad=14,fontstyle='normal')

plt.tight_layout(rect=[0.02, 0.02, 0.85, 0.95])
plt.savefig("figures/SpineXR_VQA_Abnormality_Coverage.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


# -----------------------------
# Cohen's Kappa from 2x2 matrix
# -----------------------------
def cohens_kappa_from_cm(cm):
    """
    cm: 2x2 confusion matrix
        [[a, b],
         [c, d]]
    """
    a, b = cm[0]
    c, d = cm[1]
    N = cm.sum()

    Po = (a + d) / N
    Pe = ((a + b)*(a + c) + (c + d)*(b + d)) / (N**2)
    kappa = (Po - Pe) / (1 - Pe)

    return Po, Pe, kappa


# -----------------------------
# Plot confusion-like matrix
# -----------------------------
def plot_confusion(correct_both, wrong_both,
                   mismatch_Icorr_Twrong, mismatch_Tcorr_Iwrong,
                   title, outpdf, outpng):

    cm = np.array([
        [correct_both, mismatch_Icorr_Twrong],   # India=Correct vs Thailand={Correct,Wrong}
        [mismatch_Tcorr_Iwrong, wrong_both],     # India=Wrong   vs Thailand={Correct,Wrong}
    ], dtype=float)

    cm_pct = 100 * cm / cm.sum()

    fig, ax = plt.subplots(figsize=(4.4, 3.9))
    im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)

    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm_pct[i,j]:.1f}%",
                    ha="center", va="center",
                    fontsize=10, color="black")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Thailand Correct", "Thailand Wrong"], rotation=10)
    ax.set_yticklabels(["India Correct", "India Wrong"])

    ax.set_xlabel("Thailand judgment")
    ax.set_ylabel("India judgment")
    ax.set_title(title)

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label="Percent of aligned QA pairs")

    plt.tight_layout()
    os.makedirs("figures", exist_ok=True)
    plt.savefig(outpdf, bbox_inches="tight")
    plt.savefig(outpng, dpi=300, bbox_inches="tight")
    plt.show()

    return cm


# =============================
# QUESTIONS
# =============================
Q_both_correct = 855
Q_both_wrong   = 130
Q_mis_total    = 9

# Symmetric disagreement assumption
Q_Icorr_Twrong = Q_mis_total / 2
Q_Tcorr_Iwrong = Q_mis_total - Q_Icorr_Twrong

cm_Q = plot_confusion(
    Q_both_correct,
    Q_both_wrong,
    Q_Icorr_Twrong,
    Q_Tcorr_Iwrong,
    "Questions: India vs Thailand",
    "figures/Questions_confusion_matrix.pdf",
    "figures/Questions_confusion_matrix.png"
)

Po_Q, Pe_Q, kappa_Q = cohens_kappa_from_cm(cm_Q)

print("\nQUESTIONS")
print(f"Observed agreement (Po): {Po_Q:.4f}")
print(f"Expected agreement (Pe): {Pe_Q:.4f}")
print(f"Cohen's κ: {kappa_Q:.4f}")


# =============================
# ANSWERS
# =============================
A_both_correct = 655
A_both_wrong   = 312
A_mis_total    = 27

# Symmetric disagreement assumption
A_Icorr_Twrong = A_mis_total / 2
A_Tcorr_Iwrong = A_mis_total - A_Icorr_Twrong

cm_A = plot_confusion(
    A_both_correct,
    A_both_wrong,
    A_Icorr_Twrong,
    A_Tcorr_Iwrong,
    "Answers: India vs Thailand",
    "figures/Answers_confusion_matrix.pdf",
    "figures/Answers_confusion_matrix.png"
)

Po_A, Pe_A, kappa_A = cohens_kappa_from_cm(cm_A)

print("\nANSWERS")
print(f"Observed agreement (Po): {Po_A:.4f}")
print(f"Expected agreement (Pe): {Pe_A:.4f}")
print(f"Cohen's κ: {kappa_A:.4f}")


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# same data as above
df = pd.DataFrame({
    "Relation": ["Agree (correct)", "Agree (wrong)", "Mismatch"],
    "Questions": [855, 130, 9],
    "Answers":   [655, 312, 27],
})
pct = df.set_index("Relation").apply(lambda col: 100*col/col.sum(), axis=0)

colors = {

    "Agree (correct)": "#7F9CB1",
    "Agree (wrong)":   "#C5966D",
    "Mismatch":        "#66a366",
}
outdir = "figures"; os.makedirs(outdir, exist_ok=True)

fig, ax = plt.subplots(figsize=(6.2, 2.8))
y = [0, 1]
labels = ["Questions", "Answers"]

left = np.zeros(2)
for rel in ["Agree (correct)", "Agree (wrong)", "Mismatch"]:
    vals = [pct.loc[rel, "Questions"], pct.loc[rel, "Answers"]]
    bars = ax.barh(y, vals, left=left, color=colors[rel], label=rel, edgecolor="white", linewidth=0.6)
    for i, b in enumerate(bars):
        v = vals[i]
        if v >= 2.0:
            ax.text(left[i] + v/2, b.get_y() + b.get_height()/2,
                    f"{v:.1f}%", ha="center", va="center", color="white", fontsize=9, weight="bold")
    left += vals

ax.set_xlim(0, 100)
ax.set_xlabel("Percentage of aligned pairs (%)")
ax.set_yticks(y); ax.set_yticklabels(labels)
ax.set_title("Agreement distribution on shared images (283) / aligned pairs (994)")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.2))

#ax.grid(axis="x", alpha=0.2)

#fig.tight_layout()
fig.savefig(os.path.join(outdir, "Agreement_QA_horizontal.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(outdir, "Agreement_QA_horizontal.png"), dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
# similarity_scores_tfidf_only.py
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re, unicodedata

INPUT_CSV = "converted.csv"
OUTPUT_CSV = "Similarity_Scores.csv"

def normalize_colname(s: str) -> str:
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00A0", " ").replace("\u200B", "")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 1) Load + normalize headers
df = pd.read_csv(INPUT_CSV)
df.columns = [normalize_colname(c) for c in df.columns]
print("Normalized columns:", list(df.columns))

# 2) Verify required headers
required_cols = [
    "Image_Id","Q_Category",
    "Llama_Questions","Llama_Answers",
    "Gemini_Questions","Gemini_Answers"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns after normalization: {missing}")

# 3) Clean text (keep blanks as empty strings so TF-IDF handles them)
for c in ["Llama_Questions","Gemini_Questions","Llama_Answers","Gemini_Answers"]:
    df[c] = df[c].fillna("").astype(str)

# 4) Cosine (row-wise TF-IDF)
def pairwise_row_cosine(left: pd.Series, right: pd.Series):
    # Fit on the union vocabulary of both columns
    texts = left.tolist() + right.tolist()
    vec = TfidfVectorizer(stop_words="english")
    X = vec.fit_transform(texts)
    n = len(left)
    A, B = X[:n], X[n:]
    return [cosine_similarity(A[i], B[i])[0, 0] for i in range(n)]

# Optional peek in notebooks:
# display(df["Llama_Questions"].head(5))
# display(df["Gemini_Questions"].head(5))

df["Q_Cosine"] = pairwise_row_cosine(df["Llama_Questions"], df["Gemini_Questions"])
df["A_Cosine"] = pairwise_row_cosine(df["Llama_Answers"],   df["Gemini_Answers"])

# 5) Save + summary
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved: {OUTPUT_CSV}")

print("\n📊 Summary (means):")
print(f"Q_Cosine : {df['Q_Cosine'].mean():.3f}")
print(f"A_Cosine : {df['A_Cosine'].mean():.3f}")


In [ ]:

!pip install bert-score

In [ ]:
def compute_semantic_similarity(left, right, model_name='google/embeddinggemma-300m'):
    from sentence_transformers import SentenceTransformer, util
    
    # Load Google's EmbeddingGemma model
    model = SentenceTransformer(model_name)
    
    # Use the STS (Semantic Textual Similarity) prompt for similarity tasks
    original_embeddings = model.encode(left.tolist(), 
                                     prompt_name="STS", 
                                     convert_to_tensor=True)
    perturbed_embeddings = model.encode(right.tolist(), 
                                      prompt_name="STS", 
                                      convert_to_tensor=True)
    
    # Compute cosine similarities
    cosine_similarities = util.pytorch_cos_sim(original_embeddings, perturbed_embeddings).diagonal().cpu().numpy()
    return cosine_similarities

In [ ]:
# similarity_scores_drop_empty.py
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer  # optional (lexical cosine)
from sklearn.metrics.pairwise import cosine_similarity
from bert_score import score as bertscore
import re, unicodedata
import sys

INPUT_CSV = "/Users/deepalimishra/Documents/MOE /Code_for_validated_dataset/Llama_vs_Gemini_2.5 pro.csv"
OUTPUT_CSV = "Similarity_Scores.csv"
BERT_MODEL = "microsoft/deberta-base-mnli"
BERT_LANG = "en"
RESCALE = True

REQ_TEXT_COLS = ["Llama_Questions","Gemini_Questions","Llama_Answers","Gemini_Answers"]

def normalize_colname(s: str) -> str:
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00A0", " ").replace("\u200B", "")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 1) Load + normalize headers
df = pd.read_csv(INPUT_CSV)
df.columns = [normalize_colname(c) for c in df.columns]
print("Normalized columns:", list(df.columns))

# 2) Verify required headers
required_cols = ["Image_Id","Q_Category"] + REQ_TEXT_COLS
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns after normalization: {missing}")

# 3) DROP rows with NaN/empty in ANY of the 4 text columns (after strip)
def non_empty(s: pd.Series) -> pd.Series:
    # keep only non-null AND non-blank after stripping
    return s.notna() & s.astype(str).str.strip().ne("")

mask = pd.Series(True, index=df.index)
for c in REQ_TEXT_COLS:
    mask &= non_empty(df[c])

dropped = (~mask).sum()
df_clean = df[mask].copy()
df_clean.reset_index(drop=True, inplace=True)

print(f"Rows before: {len(df)} | dropped (any empty in {REQ_TEXT_COLS}): {dropped} | kept: {len(df_clean)}")
if df_clean.empty:
    sys.exit("No rows left after dropping empties. Please inspect your data.")

# 4) Cosine (semantic similarity using SentenceTransformer embeddings)
def compute_semantic_similarity(left, right, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    from sentence_transformers import SentenceTransformer, util

    left_texts  = left.astype(str).tolist()
    right_texts = right.astype(str).tolist()

    model = SentenceTransformer(model_name)
    emb_left  = model.encode(
        left_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=32
    )
    emb_right = model.encode(
        right_texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=32
    )
    sims = util.cos_sim(emb_left, emb_right).diagonal().cpu().numpy()
    return sims

# (Optional) lexical TF-IDF cosine for comparison
def compute_lexical_cosine(left, right):
    texts = left.astype(str).tolist() + right.astype(str).tolist()
    vec = TfidfVectorizer(stop_words="english")
    X = vec.fit_transform(texts)
    n = len(left)
    A, B = X[:n], X[n:]
    return [cosine_similarity(A[i], B[i])[0, 0] for i in range(n)]

# Use SEMANTIC cosine
df_clean["Q_Cosine"] = compute_semantic_similarity(df_clean["Llama_Questions"], df_clean["Gemini_Questions"])
df_clean["A_Cosine"] = compute_semantic_similarity(df_clean["Llama_Answers"],   df_clean["Gemini_Answers"])

# 5) BERTScore (Gemini as candidate vs LLaMA as reference)
def bertscore_pair(cands, refs, model=BERT_MODEL, lang=BERT_LANG, rescale=RESCALE):
    P, R, F1 = bertscore(
        cands=list(cands.astype(str)),
        refs=list(refs.astype(str)),
        lang=lang,
        model_type=model,
        rescale_with_baseline=rescale
    )
    return [float(x) for x in P], [float(x) for x in R], [float(x) for x in F1]

qP, qR, qF = bertscore_pair(df_clean["Gemini_Questions"], df_clean["Llama_Questions"])
aP, aR, aF = bertscore_pair(df_clean["Gemini_Answers"],   df_clean["Llama_Answers"])

df_clean["Q_BERT_P"], df_clean["Q_BERT_R"], df_clean["Q_BERT_F1"] = qP, qR, qF
df_clean["A_BERT_P"], df_clean["A_BERT_R"], df_clean["A_BERT_F1"] = aP, aR, aF

# 6) Save + summary
df_clean.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved: {OUTPUT_CSV}")

print("\n📊 Summary on kept rows (means):")
print(f"Q_Cosine  : {df_clean['Q_Cosine'].mean():.3f}")
print(f"A_Cosine  : {df_clean['A_Cosine'].mean():.3f}")
print(f"Q_BERT_F1 : {df_clean['Q_BERT_F1'].mean():.3f} (P={df_clean['Q_BERT_P'].mean():.3f}, R={df_clean['Q_BERT_R'].mean():.3f})")
print(f"A_BERT_F1 : {df_clean['A_BERT_F1'].mean():.3f} (P={df_clean['A_BERT_P'].mean():.3f}, R={df_clean['A_BERT_R'].mean():.3f})")


In [ ]:
import torch
import pandas as pd
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# ==============================
# CONFIG
# ==============================
INPUT_CSV  = "/Users/deepalimishra/Documents/MOE /Code_for_validated_dataset/Llama_vs_Gemini_2.5 pro.csv"  
df = pd.read_csv(INPUT_CSV)
print(df.columns.tolist())# change path
ANSWER_COL = "Llama_Questions"
PRED_COL   = "Gemini_Questions"


MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
BATCH_SIZE = 16
MAX_LENGTH = 256

# ==============================
# Load CSV
# ==============================
df = pd.read_csv(INPUT_CSV)

df[ANSWER_COL] = df[ANSWER_COL].fillna("").astype(str)
df[PRED_COL]   = df[PRED_COL].fillna("").astype(str)

texts_a = df[ANSWER_COL].tolist()
texts_b = df[PRED_COL].tolist()

# ==============================
# Load model
# ==============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# ==============================
# Mean pooling
# ==============================
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
        input_mask_expanded.sum(1), min=1e-9
    )

# ==============================
# Batch embedding function
# ==============================
def get_embeddings(texts):
    all_embs = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            out = model(**enc)

        emb = mean_pooling(out, enc["attention_mask"])
        emb = F.normalize(emb, p=2, dim=1)  # L2 normalize
        all_embs.append(emb.cpu())

    return torch.cat(all_embs, dim=0)

# ==============================
# Compute embeddings
# ==============================
print("Computing Bio_ClinicalBERT embeddings...")
emb_a = get_embeddings(texts_a)
emb_b = get_embeddings(texts_b)

# ==============================
# Compute cosine similarity
# ==============================
sims = F.cosine_similarity(emb_a, emb_b, dim=1).numpy()

mean_score = sims.mean()
print("\n===============================")
print(" Mean Semantic Similarity Score")
print(" (Bio_ClinicalBERT) =", mean_score)
print("===============================")


In [ ]:
from pathlib import Path
import pandas as pd
from bert_score import score as bertscore_score

# ==============================
# CONFIG
# ==============================
CSV_PATH = Path("/Users/deepalimishra/Documents/MOE /Code_for_validated_dataset/Llama_vs_Gemini_2.5 pro.csv")

REF_COL  = "Llama_Answers"      # ground-truth column name
PRED_COL = "Gemini_Answers "    # prediction column name (currently with trailing space)

ERROR_PREFIX = "[ERROR]"        # set to None if you don't want this filter

BERT_MODEL_TYPE   = "bert-base-uncased"
#BERT_LANG         = "en"
BERT_BATCH_SIZE   = 64
#BERT_RESCALE_BASE = True        # recommended for English


# ==============================
# Load CSV and clean columns
# ==============================
df = pd.read_csv(CSV_PATH)

# Strip whitespace from column names so "Gemini_Answers " matches "Gemini_Answers"
df.columns = df.columns.str.strip()

# Also strip from REF_COL / PRED_COL config, just in case
REF_COL  = REF_COL.strip()
PRED_COL = PRED_COL.strip()

print("Columns in CSV:", df.columns.tolist())
if REF_COL not in df.columns:
    raise KeyError(f"Reference column '{REF_COL}' not found in CSV.")
if PRED_COL not in df.columns:
    raise KeyError(f"Prediction column '{PRED_COL}' not found in CSV.")

# ==============================
# Basic cleaning
# ==============================
df[REF_COL]  = df[REF_COL].fillna("").astype(str)
df[PRED_COL] = df[PRED_COL].fillna("").astype(str)

# Optional: filter out rows with clearly invalid predictions
if ERROR_PREFIX is not None:
    mask = ~df[PRED_COL].str.startswith(ERROR_PREFIX)
    before = len(df)
    df = df[mask].reset_index(drop=True)
    print(f"Filtered rows with predictions starting with '{ERROR_PREFIX}': {before} -> {len(df)}")

# Extract lists
refs  = df[REF_COL].tolist()
preds = df[PRED_COL].tolist()

print(f"Number of pairs for BERTScore: {len(refs)}")

# ==============================
# Compute BERTScore
# ==============================
print("Computing BERTScore...")

P, R, F1 = bertscore_score(
    cands=preds,
    refs=refs,
    model_type=BERT_MODEL_TYPE,
    #lang=BERT_LANG,
    batch_size=BERT_BATCH_SIZE,
    #rescale_with_baseline=BERT_RESCALE_BASE,
    verbose=True,
)

# Convert to Python floats
P_mean = P.mean().item()
R_mean = R.mean().item()
F1_mean = F1.mean().item()

print("\n===============================")
print(" BERTScore (Bio-style answers)")
print(f"  Precision: {P_mean:.4f}")
print(f"  Recall:    {R_mean:.4f}")
print(f"  F1:        {F1_mean:.4f}")
print("===============================")

# Optional: attach per-row scores back to CSV
df["BERTScore_P"]  = P.tolist()
df["BERTScore_R"]  = R.tolist()
df["BERTScore_F1"] = F1.tolist()

out_path = CSV_PATH.with_name(CSV_PATH.stem + "_with_bertscore.csv")
df.to_csv(out_path, index=False)
print(f"\nPer-sample BERTScore saved to: {out_path}")


In [ ]:
!pip install bert-core rouge-score -qqq

In [ ]:
# make a hist plot of Q_Cosine
plt.figure(figsize=(7,4))
plt.hist(df_clean["A_BERT_F1"], bins=30, color="#4C72B0", edgecolor="black", alpha=0.7)
plt.title("Histogram of Question Semantic Cosine Similarity")
plt.xlabel("Cosine Similarity")
plt.ylabel("Frequency")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.xlim(0, 1)
# plt.savefig("figures/Question_Semantic_Cosine_Histogram.png", dpi=300)
plt.show()

In [ ]:
df.to_csv("Similarity_Scores_with_Semantic_Cosine.csv", index=False)

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

# ============================================================
# CONFIG
# ============================================================
CSV_PATH = Path("/Users/deepalimishra/Documents/MOE /Code_for_validated_dataset/Llama_vs_Gemini_2.5 pro.csv")

# 🔴 EDIT THESE to match your CSV column names
LLAMA_Q_COL   = "Llama_Questions"    # e.g., "Llama_Questions"
GEMINI_Q_COL  = "Gemini_Questions"   # e.g., "Gemini_Questions"
LLAMA_A_COL   = "Llama_Answers"      # you already have this
GEMINI_A_COL  = "Gemini_Answers "    # note: trailing space in your file

# Similarity scores you already computed (constants)
BERT_Q_MEAN = 0.7245
BERT_A_MEAN = 0.6489

BIO_Q_MEAN_REPORTED = 0.92795354
BIO_A_MEAN_REPORTED = 0.9491458

COS_Q_MEAN = 0.661
COS_A_MEAN = 0.661

# Bio_ClinicalBERT model (same as you used)
BIO_CLINICAL_MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
BATCH_SIZE = 16
MAX_LENGTH = 256

# ============================================================
# Load CSV
# ============================================================
df = pd.read_csv(CSV_PATH)

# strip whitespace from column names (fixes "Gemini_Answers ")
df.columns = df.columns.str.strip()

LLAMA_Q_COL  = LLAMA_Q_COL.strip()
GEMINI_Q_COL = GEMINI_Q_COL.strip()
LLAMA_A_COL  = LLAMA_A_COL.strip()
GEMINI_A_COL = GEMINI_A_COL.strip()

print("Columns in CSV:", df.columns.tolist())

for col in [LLAMA_Q_COL, GEMINI_Q_COL, LLAMA_A_COL, GEMINI_A_COL]:
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found in CSV. Please check the name.")

# basic cleaning
for col in [LLAMA_Q_COL, GEMINI_Q_COL, LLAMA_A_COL, GEMINI_A_COL]:
    df[col] = df[col].fillna("").astype(str)

questions_llama  = df[LLAMA_Q_COL].tolist()
questions_gemini = df[GEMINI_Q_COL].tolist()
answers_llama    = df[LLAMA_A_COL].tolist()
answers_gemini   = df[GEMINI_A_COL].tolist()

print(f"#pairs: {len(questions_llama)}")


# ============================================================
# Bio_ClinicalBERT similarity helpers
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(BIO_CLINICAL_MODEL_NAME)
model = AutoModel.from_pretrained(BIO_CLINICAL_MODEL_NAME)
model = model.to(device)
model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
        input_mask_expanded.sum(1), min=1e-9
    )

def compute_bioclinical_similarity(texts_a, texts_b):
    """Compute per-pair cosine similarity using Bio_ClinicalBERT."""
    assert len(texts_a) == len(texts_b)
    all_scores = []

    for i in range(0, len(texts_a), BATCH_SIZE):
        batch_a = texts_a[i:i+BATCH_SIZE]
        batch_b = texts_b[i:i+BATCH_SIZE]

        enc_a = tokenizer(
            batch_a,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(device)

        enc_b = tokenizer(
            batch_b,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            out_a = model(**enc_a)
            out_b = model(**enc_b)

        emb_a = mean_pooling(out_a, enc_a["attention_mask"])
        emb_b = mean_pooling(out_b, enc_b["attention_mask"])

        # L2-normalize
        emb_a = F.normalize(emb_a, p=2, dim=1)
        emb_b = F.normalize(emb_b, p=2, dim=1)

        # cosine similarity per row
        sim = F.cosine_similarity(emb_a, emb_b, dim=1)
        all_scores.extend(sim.cpu().numpy().tolist())

    return np.array(all_scores)


# ============================================================
# Compute per-sample Bio_ClinicalBERT similarity
# ============================================================
print("Computing Bio_ClinicalBERT similarity for QUESTIONS...")
q_sim_bio = compute_bioclinical_similarity(questions_llama, questions_gemini)
print("Questions: mean =", q_sim_bio.mean())

print("Computing Bio_ClinicalBERT similarity for ANSWERS...")
a_sim_bio = compute_bioclinical_similarity(answers_llama, answers_gemini)
print("Answers: mean =", a_sim_bio.mean())

# ============================================================
# Prepare data for ACM-style plot
# ============================================================
metrics = ["BERTScore", "BioClinicalBERT", "Embedding Cosine"]

question_means = np.array([
    BERT_Q_MEAN,
    q_sim_bio.mean(),    # use actual computed mean instead of hard-coded
    COS_Q_MEAN
])

answer_means = np.array([
    BERT_A_MEAN,
    a_sim_bio.mean(),    # use actual computed mean instead of hard-coded
    COS_A_MEAN
])

# ============================================================
# Matplotlib: ACM-ish styling
# ============================================================
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 11,
    "axes.titlesize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))  # ~one-column ACM width

# ---------------- Left: grouped bar chart ----------------
ax = axes[0]
x = np.arange(len(metrics))
width = 0.38

ax.bar(x - width/2, question_means, width,
       label="Questions", edgecolor="black")
ax.bar(x + width/2, answer_means, width,
       label="Answers", edgecolor="black", hatch="//")

ax.set_ylabel("Similarity score")
ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=20, ha="right")
ax.set_ylim(0.0, 1.05)
ax.set_title("Mean semantic similarity (LLaMA vs Gemini)")
ax.legend(frameon=False)
ax.grid(axis="y", linestyle=":", linewidth=0.5)

# ---------------- Right: scatter plot (BioClinicalBERT) ----------------
ax = axes[1]

ax.scatter(q_sim_bio, a_sim_bio, s=12, alpha=0.6, edgecolors="none")

# Diagonal y=x
lims = [min(q_sim_bio.min(), a_sim_bio.min()) - 0.01,
        max(q_sim_bio.max(), a_sim_bio.max()) + 0.01]
ax.plot(lims, lims, linestyle="--", linewidth=1.0, color="black")
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel("Question similarity (BioClinicalBERT)")
ax.set_ylabel("Answer similarity (BioClinicalBERT)")
ax.set_title("Per-pair similarity alignment")
ax.grid(True, linestyle=":", linewidth=0.5)

plt.tight_layout()

# Save vector + PNG for ACM submission
out_pdf = "llama_gemini_similarity_acm.pdf"
out_png = "llama_gemini_similarity_acm.png"
plt.savefig(out_pdf, bbox_inches="tight")
plt.savefig(out_png, dpi=300, bbox_inches="tight")
print("Saved figure as:", out_pdf, "and", out_png)

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==============================
# Your similarity means
# ==============================
metrics = ["BERTScore", "BioClinicalBERT", "Embedding Cosine"]
question_means = np.array([0.7245, 0.9279535, 0.661])
answer_means   = np.array([0.6489, 0.9491458, 0.661])

# q_sim_bio and a_sim_bio already computed previously

# ==============================
# ACM-style typography
# ==============================
plt.rcParams.update({
    "figure.dpi": 250,
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.9))

# =========================================
# LEFT PANEL — bar chart
# =========================================
ax = axes[0]
x = np.arange(len(metrics))
width = 0.33

q_color = "#F4CCAB"
a_color = "#EDB4DF"

bars_q = ax.bar(
    x - width/2, question_means, width,
    color=q_color, edgecolor="black", linewidth=0.6,
    label="Questions"
)
bars_a = ax.bar(
    x + width/2, answer_means, width,
    color=a_color, edgecolor="black", linewidth=0.6,
    label="Answers"
)

ax.set_ylabel("Similarity Score")
ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=20, ha="right")
ax.set_ylim(0.0, 1.05)
ax.set_title("Mean Semantic Similarity", pad=8, loc="center")

# --- FIX legend overlap: place outside ---
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(0.01, 1.04))

# --- REMOVE GRID LINES ---
ax.grid(False)

# =========================================
# RIGHT PANEL — scatter plot
# =========================================
ax = axes[1]

ax.scatter(
    q_sim_bio, a_sim_bio,
    s=22, color="#4C72B0", alpha=0.65, edgecolors="none"
)

lims = [
    min(q_sim_bio.min(), a_sim_bio.min()) - 0.01,
    max(q_sim_bio.max(), a_sim_bio.max()) + 0.01
]
ax.plot(lims, lims, "--", linewidth=1.0, color="black")
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel("Question Similarity (BioClinicalBERT)")
ax.set_ylabel("Answer Similarity (BioClinicalBERT)")
ax.set_title("Per-Pair Similarity Alignment", loc="center")

# --- REMOVE GRID LINES ---
ax.grid(False)

plt.tight_layout()

plt.savefig("llama_gemini_similarity_acm_fixed.pdf", bbox_inches="tight")
plt.savefig("llama_gemini_similarity_acm_fixed.png", dpi=350, bbox_inches="tight")

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ==== data ====
categories = ["India-Test", "India-Train", "Thailand-Test", "Thailand-Train"]
question_wrong = [15.6, 13.8, 1.6, 4.4]
answer_wrong = [40.5, 39.1, 30.7, 29.8]

x = np.arange(len(categories))
width = 0.35

# ==== figure ====
fig, ax = plt.subplots(figsize=(10, 5))

# Bars
q_bars = ax.bar(
    x - width/2, question_wrong, width,
    label='Question wrong (%)', color='#F4C5EE'
)
a_bars = ax.bar(
    x + width/2, answer_wrong, width,
    label='Answer wrong (%)', color='#AEB1EA'
)

# Percentage labels on bars
ax.bar_label(q_bars, fmt='%.1f%%', padding=3, fontsize=10)
ax.bar_label(a_bars, fmt='%.1f%%', padding=3, fontsize=10)

# Axis labels and title
ax.set_ylabel('Error Rate (%)', fontsize=12)
ax.set_title(
    'Question vs. Answer Error Rates across Country–Split Subsets',
    fontsize=13, pad=10
)

# Ticks
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=12)
ax.tick_params(axis='y', labelsize=12)

# Legend
ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig("error_rates_plot.pdf", format="pdf", bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import os
import numpy as np

# -----------------------------
# CONFIG
# -----------------------------
MODEL_FILES = {
    "blip2_ft": "For evaluation/BLIP2_finetuned_Uannotated.xlsx",
    "claude_sonnet": "For evaluation/claude_sonnet.xlsx",
    "deepseek_vl2": "For evaluation/deepseek_vl2_predictions.xlsx",
    "lingshu_7b": "For evaluation/lingshu_7b_zeroshot_spinexr.xlsx",
    "medgemma_finetuned": "For evaluation/medgemma_finetuned_final.xlsx",
}

N_NORMAL_IMAGES = 10
N_ABNORMAL_IMAGES = 25
EXPECTED_ABNORMAL_QA = 6
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# -----------------------------
# IMAGE ID NORMALIZATION
# -----------------------------
def normalize_image_id(x):
    if pd.isna(x):
        return None
    return os.path.basename(str(x)).replace(".png", "")

# -----------------------------
# LOAD ALL MODELS
# -----------------------------
dfs = {}

for model_name, file_path in MODEL_FILES.items():
    df = pd.read_excel(file_path)

    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace("\n", "", regex=False)
        .str.replace("\xa0", "", regex=False)
    )

    df["image_id_norm"] = df["image_id"].apply(normalize_image_id)
    dfs[model_name] = df

print("✅ All model files loaded and standardized")

# -----------------------------
# STEP 1: FIND COMMON IMAGES
# -----------------------------
common_images = None
for df in dfs.values():
    imgs = set(df["image_id_norm"].dropna().unique())
    common_images = imgs if common_images is None else common_images.intersection(imgs)

print(f"🔍 Images common to all models: {len(common_images)}")

# -----------------------------
# STEP 2: CLASSIFY USING QA COUNT
# (use ONE reference model)
# -----------------------------
ref_model = list(dfs.keys())[0]
ref_df = dfs[ref_model]

normal_images = []
abnormal_images = []

for img_id in common_images:
    qa_count = ref_df[ref_df["image_id_norm"] == img_id]["question"].nunique()

    if qa_count == 1:
        normal_images.append(img_id)
    elif qa_count == EXPECTED_ABNORMAL_QA:
        abnormal_images.append(img_id)

print(f"✅ Normal images found (1 QA): {len(normal_images)}")
print(f"✅ Abnormal images found (6 QA): {len(abnormal_images)}")

# -----------------------------
# STEP 3: SAMPLE IMAGES
# -----------------------------
if len(normal_images) < N_NORMAL_IMAGES:
    raise ValueError(f"Not enough normal images ({len(normal_images)})")

if len(abnormal_images) < N_ABNORMAL_IMAGES:
    raise ValueError(f"Not enough abnormal images ({len(abnormal_images)})")

selected_images = pd.DataFrame({
    "image_id_norm": (
        list(np.random.choice(normal_images, N_NORMAL_IMAGES, replace=False)) +
        list(np.random.choice(abnormal_images, N_ABNORMAL_IMAGES, replace=False))
    )
})

# -----------------------------
# STEP 4: FETCH ALL QA PAIRS
# -----------------------------
os.makedirs("selected_images_all_qas_per_model", exist_ok=True)

for model_name, df in dfs.items():
    df_sel = df.merge(selected_images, on="image_id_norm")

    df_sel = df_sel[
        ["image_id", "image_id_norm", "question",
         "ground_truth", "generated_answer"]
    ]

    out_path = (
        f"selected_images_all_qas_per_model/"
        f"{model_name}_normal{N_NORMAL_IMAGES}_abnormal{N_ABNORMAL_IMAGES}.xlsx"
    )

    df_sel.to_excel(out_path, index=False)
    print(f"✅ Saved {out_path} ({len(df_sel)} rows)")

print("🎯 Selection complete.")
